# 08-2. 설정 우선순위와 환경 변수 예제

## Goal

- 기본값·JSON·환경 변수·CLI 우선순위를 적용합니다.
- 비밀값을 공개 설정과 분리합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

실제 환경 전체를 읽지 않고 허용된 합성 딕셔너리만 사용합니다.


## Steps

### 설정 계층 병합

뒤에 적용한 계층이 앞의 값을 덮어쓰되 `None`은 덮어쓰지 않습니다.


In [1]:
DEFAULTS = {"target": "http://127.0.0.1:8080", "timeout": 3.0, "log_level": "INFO"}


def resolve_settings(json_values, environment_values, cli_values):
    allowed = set(DEFAULTS)
    for source in (json_values, environment_values, cli_values):
        unknown = set(source) - allowed
        if unknown:
            raise ValueError(f"지원하지 않는 설정: {sorted(unknown)}")
    result = dict(DEFAULTS)
    for source in (json_values, environment_values, cli_values):
        result.update({key: value for key, value in source.items() if value is not None})
    return result


settings = resolve_settings(
    {"timeout": 5.0},
    {"log_level": "WARNING"},
    {"timeout": 1.5, "target": None},
)
print(settings)


{'target': 'http://127.0.0.1:8080', 'timeout': 1.5, 'log_level': 'WARNING'}


## Checks

CLI 값이 최종 우선순위를 가지며 미지정 값은 기본값을 보존하는지 확인합니다.


In [2]:
assert settings == {"target": "http://127.0.0.1:8080", "timeout": 1.5, "log_level": "WARNING"}
try:
    resolve_settings({}, {}, {"api_token": "값"})
except ValueError:
    print("허용되지 않은 설정 거부 확인")


허용되지 않은 설정 거부 확인


## Next Steps

API 토큰은 설정 보고서와 로그에 포함하지 않고 필요한 코드 경계에만 전달합니다.
